# 2. Training: RLCD with a teacher

`layaft train` follows the recipe of [Laya's official notebook](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb)
on one GPU:

1. **Cases** from `data/<task>.jsonl`.
2. **Teacher.** Jev answers the same questions on every case (cached in `data/<task>_teacher.jsonl`). Its
   distribution softens the target (30 %) and the cases where it disagrees with a `choice` answer are dropped.
   With `teacher="none"`, or without a key, the label is smoothed to 90 %.
3. **Split** 80 / 10 / 10: train, calibration, validation. The hand-written test set is never trained on.
4. **Context.** Above the checkpoint's positions the encoder is extended with YaRN first; above 2048 tokens, long
   copies of the cases (the case inside neutral filler) join the short ones.
5. **RLCD**, **temperature calibration** per question type, comparison, and a checkpoint in Laya's format under
   `runs/`, checked by reloading it with `laya.load`.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # data/, tasks/ and runs/ live in the repo root.
from layaft import LayaFT, Task

task = Task.load("helpdesk")

import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU: training needs CUDA")

## Short test (≤ 3 GB of GPU)

Checks the whole loop on a laptop: 2 cases per combination, one epoch, the top 6 encoder layers.

In [ ]:
LayaFT("multilingual").train(task, profile="test")

## Full training, and the context ladder

On a 16 GB GPU every epoch over ~12,000 short cases takes minutes. Each context stage starts from the previous one.

In [ ]:
m = LayaFT("multilingual")
m.train(task, profile="full")                 # 1024 tokens, the checkpoint's own
# m.train(task, profile="full", ctx="8k")     # native positions of mmBERT, now trained on long states
# m.train(task, profile="full", ctx="32k")    # YaRN ×4 on the global-attention layers

## The report saved next to the checkpoint

In [ ]:
import json
from IPython.display import Markdown, display
from layaft.evaluate import report

r = json.loads((Path(m.model) / "results.json").read_text())
print({k: r[k] for k in ("ctx", "cases", "best_validation_ce", "max_vram_gb")})
display(Markdown(report.table(task, {"test: base": r["base"]["test"], "test: tuned": r["tuned"]["test"]})))